In [ ]:
%pip install neo4j datasets google-genai pydantic python-dotenv numpy tqdm --quiet

In [ ]:
import os
import pickle
import sys
from pathlib import Path

from datasets import load_dataset
from dotenv import load_dotenv

load_dotenv("../.env")

sys.path.insert(0, str(Path("..").resolve()))

In [ ]:
from qasa_rag import KnowledgeGraphLoader

In [ ]:
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password123")

In [ ]:
raw_dataset = load_dataset("dgslibisey/MuSiQue", split="validation")
print(f"Total validation examples: {len(raw_dataset):,}")

answerable_dataset = raw_dataset.filter(lambda x: x["answerable"])
print(f"Answerable examples: {len(answerable_dataset):,}")

dataset = answerable_dataset.select(range(1000))
print(f"Selected {len(dataset)} examples")

In [ ]:
def transform_musique_to_loader_format(example):
    """Transform MuSiQue format to match loader expectations.

    Output format per example::

        {
            "question": str,
            "answer": str,
            "paragraphs": [{"title": str, "text": str, "is_supporting": bool}, ...],
        }
    """
    return {
        "question": example["question"],
        "answer": example["answer"],
        "paragraphs": [
            {
                "title": para["title"],
                "text": para["paragraph_text"],
                "is_supporting": para["is_supporting"],
            }
            for para in example["paragraphs"]
        ],
    }

transformed_dataset = [transform_musique_to_loader_format(example) for example in dataset]
print(f"Transformed {len(transformed_dataset)} examples")
print(f"Q: {transformed_dataset[0]['question']}")
print(f"A: {transformed_dataset[0]['answer']}")
print(f"Paragraphs: {len(transformed_dataset[0]['paragraphs'])}")

In [ ]:
loader = KnowledgeGraphLoader(
    neo4j_uri=NEO4J_URI,
    neo4j_user=NEO4J_USER,
    neo4j_password=NEO4J_PASSWORD,
    cache_dir=Path("cache"),
)

In [ ]:
loader.clear_and_init()

In [ ]:
ground_truth = loader.load_examples(
    transformed_dataset,
    max_extraction_workers=30,
    max_embedding_workers=5,
    save_every=1000,
)

In [ ]:
loader.finalize()

In [ ]:
with open("ground_truth-musique.pkl", "wb") as f:
    pickle.dump(ground_truth, f)

print(f"Saved ground truth for {len(ground_truth)} questions")

In [ ]:
stats = loader.get_stats()
for key, value in stats.items():
    print(f"{key}: {value}")

In [ ]:
loader.close()